### BiLSTM-CRF for NER

- Dataset:  word_bio_flat.csv  (resume_id | token | tag)
- Tags:     BIO scheme over 10 entity types
- Model:    Embedding -> BiLSTM -> Linear -> CRF


In [2]:
import numpy as np
import pickle
import json
from pathlib import Path
from collections import defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

Device: cpu
PyTorch: 2.12.0+cpu


In [4]:
DATA_DIR = Path('../../datasets/processed_data')

# Load numpy arrays
X_tokens = np.load(DATA_DIR / 'X_tokens.npy')           # (N, 256)
X_chars  = np.load(DATA_DIR / 'X_chars.npy')            # (N, 256, 20)
Y_labels = np.load(DATA_DIR / 'Y_labels.npy')           # (N, 256)
masks    = np.load(DATA_DIR / 'masks.npy')               # (N, 256)
emb_mat  = np.load(DATA_DIR / 'embedding_matrix.npy')   # (vocab, 300)

# Load vocabularies
with open(DATA_DIR / 'vocabs.pkl', 'rb') as f:
    vocabs = pickle.load(f)

token2id      = vocabs['token2id']
id2token      = vocabs['id2token']
label2id      = vocabs['label2id']
id2label      = vocabs['id2label']
char2id       = vocabs['char2id']
PAD_ID        = vocabs['PAD_ID']
UNK_ID        = vocabs['UNK_ID']
PAD_LABEL_ID  = vocabs['PAD_LABEL_ID']
EMBED_DIM     = vocabs['EMBED_DIM']       # 300
MAX_LEN       = vocabs['MAX_LEN_LSTM']    # 256
Entity_labels = vocabs['Entity_labels']

VOCAB_SIZE  = len(token2id)
CHAR_SIZE   = len(char2id)
NUM_LABELS  = len(label2id)
MAX_CHAR    = X_chars.shape[2]            # 20

print(f'Resumes        : {X_tokens.shape[0]}')
print(f'Max seq length : {MAX_LEN}')
print(f'Token vocab    : {VOCAB_SIZE:,}')
print(f'Char vocab     : {CHAR_SIZE}')
print(f'Num labels     : {NUM_LABELS}')
print(f'Embed dim      : {EMBED_DIM}')
print(f'Label map      :')
for lbl, idx in label2id.items():
    print(f'  {idx:>3}  {lbl}')

Resumes        : 220
Max seq length : 256
Token vocab    : 4,638
Char vocab     : 62
Num labels     : 22
Embed dim      : 300
Label map      :
    0  <PAD_LABEL>
    1  O
    2  B-Name
    3  I-Name
    4  B-Designation
    5  I-Designation
    6  B-Companies worked at
    7  I-Companies worked at
    8  B-Location
    9  I-Location
   10  B-Email Address
   11  I-Email Address
   12  B-College Name
   13  I-College Name
   14  B-Degree
   15  I-Degree
   16  B-Graduation Year
   17  I-Graduation Year
   18  B-Skills
   19  I-Skills
   20  B-Years of Experience
   21  I-Years of Experience


In [5]:
class ResumeNERDataset(Dataset):
    """
    PyTorch Dataset for resume NER.
    Each item: (token_ids, char_ids, label_ids, mask)
    """
    def __init__(self, X_tokens, X_chars, Y_labels, masks):
        self.X_tokens = torch.tensor(X_tokens, dtype=torch.long)
        self.X_chars  = torch.tensor(X_chars,  dtype=torch.long)
        self.Y_labels = torch.tensor(Y_labels, dtype=torch.long)
        self.masks    = torch.tensor(masks,    dtype=torch.bool)

    def __len__(self):
        return len(self.X_tokens)

    def __getitem__(self, idx):
        return (
            self.X_tokens[idx],
            self.X_chars[idx],
            self.Y_labels[idx],
            self.masks[idx],
        )


# Train / Val / Test split  80 / 10 / 10
dataset  = ResumeNERDataset(X_tokens, X_chars, Y_labels, masks)
n        = len(dataset)
n_train  = int(0.8 * n)
n_val    = int(0.1 * n)
n_test   = n - n_train - n_val

train_ds, val_ds, test_ds = random_split(
    dataset, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(42)
)

BATCH_SIZE = 32

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

print(f'Train: {n_train} | Val: {n_val} | Test: {n_test}')

Train: 176 | Val: 22 | Test: 22


In [6]:
class CRF(nn.Module):
    """
    Conditional Random Field layer.

    Learns transition scores between BIO labels so that
    invalid tag sequences (e.g. I-Skill after O) are penalised
    during training and cannot appear in Viterbi decoding.

    Parameters
    ----------
    num_tags     : number of output labels (including PAD)
    pad_tag_id   : label id for <PAD_LABEL> — excluded from transitions
    """

    def __init__(self, num_tags: int, pad_tag_id: int = 0):
        super().__init__()
        self.num_tags   = num_tags
        self.pad_tag_id = pad_tag_id

        # Transition matrix:  transitions[i, j] = score of going from tag j -> tag i
        self.transitions = nn.Parameter(torch.randn(num_tags, num_tags))

        # Start / end pseudo-scores
        self.start_transitions = nn.Parameter(torch.randn(num_tags))
        self.end_transitions   = nn.Parameter(torch.randn(num_tags))

        self._init_transitions()

    def _init_transitions(self):
        # Forbid transitioning to/from PAD
        nn.init.constant_(self.transitions[:, self.pad_tag_id], -10000.)
        nn.init.constant_(self.transitions[self.pad_tag_id, :], -10000.)
        nn.init.constant_(self.start_transitions[self.pad_tag_id], -10000.)
        nn.init.constant_(self.end_transitions[self.pad_tag_id],   -10000.)

    # ------------------------------------------------------------------
    # Forward score  (log-partition function via forward algorithm)
    # ------------------------------------------------------------------
    def _forward_alg(self, emissions: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        """
        emissions : (batch, seq_len, num_tags)
        mask      : (batch, seq_len)  bool — True for real tokens
        Returns   : (batch,)  log-partition
        """
        batch, seq_len, num_tags = emissions.shape
        # Alpha at t=0
        alpha = self.start_transitions + emissions[:, 0]   # (batch, num_tags)

        for t in range(1, seq_len):
            # (batch, num_tags, 1) + (num_tags, num_tags) + (batch, 1, num_tags)
            score = alpha.unsqueeze(2) + self.transitions + emissions[:, t].unsqueeze(1)
            score = torch.logsumexp(score, dim=1)          # (batch, num_tags)
            # Only update for real (non-padding) positions
            alpha = torch.where(mask[:, t].unsqueeze(1), score, alpha)

        alpha = alpha + self.end_transitions
        return torch.logsumexp(alpha, dim=1)               # (batch,)

    # ------------------------------------------------------------------
    # Gold score  (sum of emission + transition scores along true path)
    # ------------------------------------------------------------------
    def _score_sentence(self, emissions: torch.Tensor,
                         tags: torch.Tensor,
                         mask: torch.Tensor) -> torch.Tensor:
        batch, seq_len, _ = emissions.shape

        # Start score
        score = self.start_transitions[tags[:, 0]] + emissions[:, 0].gather(1, tags[:, 0:1]).squeeze(1)

        for t in range(1, seq_len):
            active = mask[:, t]                             # (batch,)
            trans  = self.transitions[tags[:, t], tags[:, t - 1]]   # (batch,)
            emit   = emissions[:, t].gather(1, tags[:, t:t+1]).squeeze(1)
            score  = score + torch.where(active, trans + emit, torch.zeros_like(score))

        # End score — use last real tag
        seq_ends = mask.long().sum(dim=1) - 1              # index of last real token
        last_tags = tags.gather(1, seq_ends.unsqueeze(1)).squeeze(1)
        score = score + self.end_transitions[last_tags]
        return score                                        # (batch,)

    # ------------------------------------------------------------------
    # Viterbi decode
    # ------------------------------------------------------------------
    def decode(self, emissions: torch.Tensor, mask: torch.Tensor) -> list[list[int]]:
        """
        emissions : (batch, seq_len, num_tags)
        mask      : (batch, seq_len)  bool
        Returns   : list of lists, each containing predicted tag ids for real tokens only
        """
        batch, seq_len, num_tags = emissions.shape
        viterbi  = self.start_transitions + emissions[:, 0]    # (batch, num_tags)
        backptr  = []

        for t in range(1, seq_len):
            score = viterbi.unsqueeze(2) + self.transitions     # (batch, num_tags, num_tags)
            best_scores, best_tags = score.max(dim=1)           # (batch, num_tags)
            backptr.append(best_tags)
            cur = best_scores + emissions[:, t]
            viterbi = torch.where(mask[:, t].unsqueeze(1), cur, viterbi)

        viterbi = viterbi + self.end_transitions
        _, best_last = viterbi.max(dim=1)                       # (batch,)

        # Backtrack
        best_paths = []
        seq_lens   = mask.long().sum(dim=1).tolist()
        best_last  = best_last.tolist()

        for b in range(batch):
            best_tag = best_last[b]
            path = [best_tag]
            for bp in reversed(backptr[:seq_lens[b] - 1]):
                best_tag = bp[b, best_tag].item()
                path.append(best_tag)
            path.reverse()
            best_paths.append(path)

        return best_paths

    # ------------------------------------------------------------------
    # Negative log-likelihood loss
    # ------------------------------------------------------------------
    def forward(self, emissions: torch.Tensor,
                tags: torch.Tensor,
                mask: torch.Tensor) -> torch.Tensor:
        """
        Returns mean NLL loss over the batch.
        """
        log_likelihood = (
            self._score_sentence(emissions, tags, mask)
            - self._forward_alg(emissions, mask)
        )
        return -log_likelihood.mean()


print('CRF layer defined.')

CRF layer defined.


In [7]:
class CharCNN(nn.Module):
    """
    Character-level CNN that produces a fixed-size embedding per token.

    Input  : (batch * seq_len, max_char)  — character IDs per token
    Output : (batch * seq_len, out_channels)  — char embedding per token

    Architecture
    ------------
    Char embedding → Conv1d (kernel 3) → ReLU → MaxPool over time → dropout
    """

    def __init__(self, char_vocab_size: int, char_embed_dim: int = 30,
                 out_channels: int = 50, kernel_size: int = 3,
                 dropout: float = 0.5, pad_idx: int = 0):
        super().__init__()
        self.char_embed = nn.Embedding(char_vocab_size, char_embed_dim, padding_idx=pad_idx)
        self.conv       = nn.Conv1d(char_embed_dim, out_channels, kernel_size, padding=kernel_size // 2)
        self.dropout    = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x : (batch * seq_len, max_char)
        """
        emb = self.char_embed(x)            # (B*L, max_char, char_dim)
        emb = self.dropout(emb)
        emb = emb.permute(0, 2, 1)         # (B*L, char_dim, max_char)  for Conv1d
        conv_out = torch.relu(self.conv(emb))   # (B*L, out_channels, max_char)
        pooled   = conv_out.max(dim=2).values   # (B*L, out_channels)  global max-pool
        return pooled


print('CharCNN defined.')

CharCNN defined.


In [8]:
class BiLSTM_CRF(nn.Module):
    """
    BiLSTM-CRF for sequence labelling.

    Pipeline
    --------
    1. Token IDs  → pre-trained FastText word embeddings (optionally fine-tuned)
    2. Char IDs   → CharCNN → char embeddings
    3. Concat word + char embeddings
    4. Dropout
    5. Bidirectional LSTM  (optionally stacked)
    6. Linear projection → emission scores  (batch, seq_len, num_tags)
    7. CRF  (training: NLL loss;  inference: Viterbi decode)

    Parameters
    ----------
    embedding_matrix : np.ndarray  (vocab_size, embed_dim)
    char_vocab_size  : int
    num_tags         : int
    pad_tag_id       : int   — PAD label id
    hidden_dim       : int   — LSTM hidden size per direction
    num_layers       : int   — stacked LSTM layers
    char_embed_dim   : int
    char_out_channels: int   — CNN output channels per token
    dropout          : float
    freeze_embeddings: bool  — freeze FastText weights
    """

    def __init__(
        self,
        embedding_matrix: np.ndarray,
        char_vocab_size : int,
        num_tags        : int,
        pad_token_id    : int = 0,
        pad_tag_id      : int = 0,
        hidden_dim      : int = 256,
        num_layers      : int = 2,
        char_embed_dim  : int = 30,
        char_out_channels: int = 50,
        kernel_size     : int = 3,
        dropout         : float = 0.5,
        freeze_embeddings: bool = True,
    ):
        super().__init__()

        vocab_size, embed_dim = embedding_matrix.shape

        # ── Word embeddings (FastText pre-trained) ──────────────────────
        self.word_embed = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix, dtype=torch.float32),
            freeze=freeze_embeddings,
            padding_idx=pad_token_id,
        )

        # ── Character CNN ───────────────────────────────────────────────
        self.char_cnn = CharCNN(
            char_vocab_size = char_vocab_size,
            char_embed_dim  = char_embed_dim,
            out_channels    = char_out_channels,
            kernel_size     = kernel_size,
            dropout         = dropout,
        )

        lstm_input_dim = embed_dim + char_out_channels

        # ── Bi-LSTM ──────────────────────────────────────────────────────
        self.lstm = nn.LSTM(
            input_size    = lstm_input_dim,
            hidden_size   = hidden_dim,
            num_layers    = num_layers,
            batch_first   = True,
            bidirectional = True,
            dropout       = dropout if num_layers > 1 else 0.0,
        )

        self.dropout    = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(hidden_dim * 2)

        # ── Emission projection ──────────────────────────────────────────
        self.fc = nn.Linear(hidden_dim * 2, num_tags)

        # ── CRF ───────────────────────────────────────────────────────────
        self.crf = CRF(num_tags, pad_tag_id=pad_tag_id)

    # ------------------------------------------------------------------
    def _get_emissions(self, token_ids: torch.Tensor,
                        char_ids: torch.Tensor) -> torch.Tensor:
        batch, seq_len, max_char = char_ids.shape

        # Word embeddings
        word_emb = self.word_embed(token_ids)          # (B, L, embed_dim)
        word_emb = self.dropout(word_emb)

        # Char embeddings via CNN
        char_flat = char_ids.view(batch * seq_len, max_char)   # (B*L, max_char)
        char_emb  = self.char_cnn(char_flat)                   # (B*L, char_out)
        char_emb  = char_emb.view(batch, seq_len, -1)          # (B, L, char_out)

        # Concatenate
        combined = torch.cat([word_emb, char_emb], dim=2)      # (B, L, lstm_in)

        # BiLSTM
        lstm_out, _ = self.lstm(combined)                       # (B, L, 2*hidden)
        lstm_out    = self.layer_norm(lstm_out)
        lstm_out    = self.dropout(lstm_out)

        # Emission scores
        emissions = self.fc(lstm_out)                           # (B, L, num_tags)
        return emissions

    # ------------------------------------------------------------------
    def forward(self, token_ids, char_ids, tags, mask):
        """
        Training step — returns CRF NLL loss.
        """
        emissions = self._get_emissions(token_ids, char_ids)
        loss      = self.crf(emissions, tags, mask)
        return loss

    def predict(self, token_ids, char_ids, mask):
        """
        Inference — returns list of predicted tag-id sequences (real tokens only).
        """
        emissions = self._get_emissions(token_ids, char_ids)
        return self.crf.decode(emissions, mask)


print('BiLSTM-CRF model defined.')

BiLSTM-CRF model defined.


In [9]:
model = BiLSTM_CRF(
    embedding_matrix  = emb_mat,
    char_vocab_size   = CHAR_SIZE,
    num_tags          = NUM_LABELS,
    pad_token_id      = PAD_ID,
    pad_tag_id        = PAD_LABEL_ID,
    hidden_dim        = 256,
    num_layers        = 2,
    char_embed_dim    = 30,
    char_out_channels = 50,
    kernel_size       = 3,
    dropout           = 0.5,
    freeze_embeddings = True,   # fine-tune: set False
).to(DEVICE)

# Count parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(model)
print(f'\nTrainable params : {trainable:,}')
print(f'Total params     : {total:,}')

BiLSTM_CRF(
  (word_embed): Embedding(4638, 300, padding_idx=0)
  (char_cnn): CharCNN(
    (char_embed): Embedding(62, 30, padding_idx=0)
    (conv): Conv1d(30, 50, kernel_size=(3,), stride=(1,), padding=(1,))
    (dropout): Dropout(p=0.5, inplace=False)
  )
  (lstm): LSTM(350, 256, num_layers=2, batch_first=True, dropout=0.5, bidirectional=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
  (fc): Linear(in_features=512, out_features=22, bias=True)
  (crf): CRF()
)

Trainable params : 2,841,392
Total params     : 4,232,792


In [12]:
LR         = 1e-3
WEIGHT_DECAY = 1e-4
NUM_EPOCHS  = 30
PATIENCE    = 5      # early stopping

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer)

print(f'Optimizer : AdamW  lr={LR}  wd={WEIGHT_DECAY}')
print(f'Scheduler : ReduceLROnPlateau  factor=0.5  patience=2')
print(f'Max epochs: {NUM_EPOCHS}  |  Early stopping patience: {PATIENCE}')

Optimizer : AdamW  lr=0.001  wd=0.0001
Scheduler : ReduceLROnPlateau  factor=0.5  patience=2
Max epochs: 30  |  Early stopping patience: 5


In [16]:
def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    for token_ids, char_ids, labels, mask in loader:
        token_ids = token_ids.to(device)
        char_ids  = char_ids.to(device)
        labels    = labels.to(device)
        mask      = mask.to(device)

        optimizer.zero_grad()
        loss = model(token_ids, char_ids, labels, mask)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def eval_epoch(model, loader, device):
    model.eval()
    total_loss = 0.0
    for token_ids, char_ids, labels, mask in loader:
        token_ids = token_ids.to(device)
        char_ids  = char_ids.to(device)
        labels    = labels.to(device)
        mask      = mask.to(device)
        loss = model(token_ids, char_ids, labels, mask)
        total_loss += loss.item()
    return total_loss / len(loader)


# ── Training loop ────────────────────────────────────────────────────
history       = {'train_loss': [], 'val_loss': []}
best_val_loss = float('inf')
patience_cnt  = 0
CKPT_PATH     = 'model_result/bilstm_crf.pt'

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_epoch(model, train_loader, optimizer, DEVICE)
    val_loss   = eval_epoch(model,  val_loader,   DEVICE)
    scheduler.step(val_loss)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)

    improved = val_loss < best_val_loss
    if improved:
        best_val_loss = val_loss
        patience_cnt  = 0
        torch.save(model.state_dict(), CKPT_PATH)
    else:
        patience_cnt += 1

    mark = ' *' if improved else ''
    print(f'Epoch {epoch:3d}/{NUM_EPOCHS}  '
          f'train={train_loss:.4f}  val={val_loss:.4f}{mark}')

    if patience_cnt >= PATIENCE:
        print(f'Early stopping at epoch {epoch}')
        break

print(f'\nBest val loss: {best_val_loss:.4f}  (checkpoint: {CKPT_PATH})')

Epoch   1/30  train=32.4054  val=38.2396 *
Epoch   2/30  train=28.1959  val=41.3103
Epoch   3/30  train=32.5190  val=38.5856
Epoch   4/30  train=36.4726  val=37.3633 *
Epoch   5/30  train=28.8325  val=40.9630
Epoch   6/30  train=28.8986  val=52.8129
Epoch   7/30  train=27.9026  val=36.4833 *
Epoch   8/30  train=25.5886  val=40.4812
Epoch   9/30  train=24.3142  val=35.8327 *
Epoch  10/30  train=25.4859  val=37.2975
Epoch  11/30  train=24.3881  val=35.1800 *
Epoch  12/30  train=21.9541  val=33.9842 *
Epoch  13/30  train=21.6422  val=35.0776
Epoch  14/30  train=19.3552  val=38.7338
Epoch  15/30  train=19.6658  val=35.1920
Epoch  16/30  train=17.4546  val=37.3879
Epoch  17/30  train=20.2326  val=33.4773 *
Epoch  18/30  train=16.9268  val=33.8894
Epoch  19/30  train=15.2830  val=34.3713
Epoch  20/30  train=14.6139  val=37.6285
Epoch  21/30  train=14.7197  val=34.8063
Epoch  22/30  train=12.3816  val=53.2280
Early stopping at epoch 22

Best val loss: 33.4773  (checkpoint: model_result/bilstm